# ISLES'24 unpack, split and survey (Phase 1)

One long session, in this order so nobody looks at images of cases that later become held-out:
1. Make a working copy of the files we use and save it to Drive, so it never has to be decoded again.
2. Check the known anomalies, in numbers only.
3. Generate the fixed split once.
4. Check headers for all cases, then spot-check alignment and labels on development cases only.
5. Measure what each candidate grid would cost.

`raw/` is not changed. No held-out ID is printed, saved in a report or drawn. The logic is in `src/strokecopilot/data/`.

**Before you start**
- Use a CPU runtime: Runtime → Change runtime type → None. A GPU runtime may have less disk.
- Drive needs about 11 GiB free for the working copy, plus 1 GiB for results. A shared folder counts against its owner's storage.
- Keep the tab open and the computer awake. Section 3 is the slow part: the first run has to copy and decode the whole archive.

**Run it:** Runtime → Run all, and allow Drive access when asked. Drive may ask again in sections 3, 5 and 9.

**If it stops:** reconnect and use Runtime → Run all. Once section 3 has saved the working copy, a rerun takes minutes instead of hours. The split is never regenerated.

**Send back:** the timing lines from section 3, the `OK:` line from section 9, and any error in full. The results are saved to `artifacts/isles24_survey/`, where Claude can read them.

## 1. Settings

In [ ]:
ROOT = '/content/drive/MyDrive/StrokeCopilot'
ARCHIVE = f'{ROOT}/raw/isles24_train.7z'
WORK_TAR = f'{ROOT}/artifacts/isles24_work/isles24_work.tar'
OUT_DIR = f'{ROOT}/artifacts/isles24_survey'
FOLDS_ON_DRIVE = f'{ROOT}/artifacts/splits/folds.json'
LOCAL_DIR = '/content/isles24'
ARCHIVE_SIZE = 99022114670
UNPACK_SECONDS = 4 * 3600
EXPECTED_CASES = 149
N_HELDOUT = 20
STRAY_CASE = 'sub-stroke0142'
SPOT_CHECK_CASES = 6
GRID_CASES = 10
GRID_SPACINGS = [1.0, 1.5, 2.0]
REPO_REF = 'main'

## 2. Install the package and mount Drive
From the public repository, at `REPO_REF`; no token is needed.

In [ ]:
import hashlib, json, os, shutil, subprocess, sys, time
from google.colab import drive

url = f'strokecopilot[isles] @ git+https://github.com/parsa-hke/strokecopilot.git@{REPO_REF}'
result = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', url], capture_output=True, text=True)
if result.returncode:
    raise RuntimeError(result.stderr[-800:])
del url, result
print('Package installed.')

drive.mount('/content/drive')
before = os.stat(ARCHIVE)
if before.st_size != ARCHIVE_SIZE:
    raise ValueError(f'Archive is {before.st_size} bytes, expected {ARCHIVE_SIZE}. Stop and check it.')
if shutil.which('7z') is None:
    subprocess.run(['apt-get', '-qq', 'install', '-y', 'p7zip-full'], check=True, capture_output=True)

from strokecopilot.data import archive_listing as al
from strokecopilot.data import isles24_layout as layout
from strokecopilot.data import isles24_survey as survey
from strokecopilot.data import splits

GIB = 1024 ** 3
os.makedirs(LOCAL_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)
results = {}
print(f'Local disk free: {shutil.disk_usage(LOCAL_DIR).free / GIB:.1f} GiB. Drive free: {shutil.disk_usage(ROOT).free / GIB:.1f} GiB.')

## 3. Working copy
First run: copies the archive to local disk (DEC-3), decodes it keeping only the files we use (about 10 GiB of 96), saves those to Drive as one tar, and checks the saved size. Timings are unmeasured. The one clue is 129 s for 1.7 GiB in the last notebook, so allow 1–3 hours in all.

Later runs copy the tar back from Drive instead.

In [ ]:
train_dir = os.path.join(LOCAL_DIR, 'train')
if not os.path.isdir(train_dir):
    started = time.time()
    if os.path.exists(WORK_TAR):
        print('Working copy found on Drive. Copying it to local disk.')
        local_tar = os.path.join(LOCAL_DIR, 'isles24_work.tar')
        print(f'Copied in {al.copy_with_progress(WORK_TAR, local_tar):.0f} s.')
        layout.unpack(local_tar, LOCAL_DIR)
        os.remove(local_tar)
    else:
        if shutil.disk_usage(LOCAL_DIR).free < 125 * GIB:
            raise OSError('Under 125 GiB free on local disk. Use a CPU runtime.')
        if shutil.disk_usage(ROOT).free < 12 * GIB:
            raise OSError('Under 12 GiB free on Drive for the working copy and results.')
        local_archive = os.path.join(LOCAL_DIR, 'isles24_train.7z')
        print(f'Archive copied in {al.copy_with_progress(ARCHIVE, local_archive):.0f} s.', flush=True)
        extract_dir = os.path.join(LOCAL_DIR, 'extracting')
        shutil.rmtree(extract_dir, ignore_errors=True)
        os.makedirs(extract_dir)
        seconds = al.extract_members(local_archive, extract_dir, layout.WORKING_MEMBERS, exclude=layout.WORKING_EXCLUDE, timeout_s=UNPACK_SECONDS)
        print(f'Decoded in {seconds:.0f} s.', flush=True)
        os.remove(local_archive)
        local_tar = os.path.join(LOCAL_DIR, 'isles24_work.tar')
        layout.pack(extract_dir, local_tar)
        os.makedirs(os.path.dirname(WORK_TAR), exist_ok=True)
        print(f'Saved to Drive in {al.copy_with_progress(local_tar, WORK_TAR):.0f} s.', flush=True)
        size = os.path.getsize(local_tar)
        drive.flush_and_unmount()  # a file seems to upload only once it is closed and flushed
        drive.mount('/content/drive', force_remount=True)
        if os.path.getsize(WORK_TAR) != size:
            os.remove(WORK_TAR)
            raise OSError('The tar on Drive is the wrong size, so it was removed. Run this section again.')
        os.replace(os.path.join(extract_dir, 'train'), train_dir)
        os.remove(local_tar)
        shutil.rmtree(extract_dir)
    print(f'Working copy ready in {time.time() - started:.0f} s.')

cases = layout.find_cases(train_dir)
missing = layout.missing_files(train_dir, cases)
if len(cases) != EXPECTED_CASES or missing:
    raise ValueError(f'Found {len(cases)} cases, expected {EXPECTED_CASES}. Missing files per type: {missing}')
results['work_tar_bytes'] = os.path.getsize(WORK_TAR)
print(f'{len(cases)} cases, none missing a file. Tar on Drive: {results["work_tar_bytes"] / GIB:.2f} GiB. Local disk free: {shutil.disk_usage(LOCAL_DIR).free / GIB:.1f} GiB.')

## 4. Known anomalies
The extra rows in two workbooks and the extra lesion mask. Numbers only, and before any split exists.

In [ ]:
tables, missing_workbooks = survey.read_clinical_tables(train_dir, cases)
if missing_workbooks:
    raise ValueError(f'{len(missing_workbooks)} clinical workbooks are missing.')
results['extra_rows'] = survey.extra_rows(tables)
results['stray_mask'] = survey.stray_mask_check(train_dir, STRAY_CASE)
print(json.dumps({k: results[k] for k in ('extra_rows', 'stray_mask')}, indent=1))

## 5. The split
Generated once and saved straight to Drive. A rerun loads the saved file and checks that the same input still gives the same split. Only group sizes and balance are printed.

In [ ]:
volumes, mask_values = survey.infarct_volumes(train_dir, cases)
centres = survey.centres_of(tables)
new_folds = splits.make_splits(volumes, centres, n_heldout=N_HELDOUT)

if os.path.exists(FOLDS_ON_DRIVE):
    folds = splits.load_folds(FOLDS_ON_DRIVE)
    if folds != new_folds:
        raise ValueError('The saved split differs from what this input gives. Stop: the split must not change.')
    print('Split already on Drive, and it matches.')
else:
    os.makedirs(os.path.dirname(FOLDS_ON_DRIVE), exist_ok=True)
    splits.write_folds(new_folds, FOLDS_ON_DRIVE)
    folds = new_folds
    drive.flush_and_unmount()
    drive.mount('/content/drive', force_remount=True)
    if splits.load_folds(FOLDS_ON_DRIVE) != folds:
        raise ValueError('The split read back from Drive differs. Stop and check it.')
    print('Split generated and saved to Drive.')

splits.validate_folds(folds, expected_cases=set(cases))
results['folds_sha256'] = hashlib.sha256(open(FOLDS_ON_DRIVE, 'rb').read()).hexdigest()
results['mask_values_cases'] = mask_values
results['zero_volume_masks'] = sum(v == 0 for v in volumes.values())
results['split_balance'] = survey.split_balance(folds, volumes, centres)
with open(os.path.join(LOCAL_DIR, 'infarct_volumes.json'), 'w') as f:
    json.dump(volumes, f, indent=1)
del new_folds
development = splits.development_ids(folds)
print('sha256', results['folds_sha256'])
print('Masks with zero volume:', results['zero_volume_masks'])
print(json.dumps(results['split_balance'], indent=1))

## 6. Headers and alignment
Headers of all cases, as counts. Alignment on development cases only. The figure shows NCCT, DWI and Tmax with the infarct outlined in red, and the Circle of Willis labels with the occlusion mask in black.

In [ ]:
results['headers'] = survey.header_survey(train_dir, cases, printable=development)
results['lesion_in_tissue'] = survey.lesion_in_tissue(train_dir, development)
print(json.dumps(results['lesion_in_tissue'], indent=1))

spot_cases = survey.sample_by_volume(volumes, development, SPOT_CHECK_CASES)
figure = os.path.join(LOCAL_DIR, 'spot_check.png')
survey.montage(train_dir, spot_cases, figure)
try:
    from IPython.display import Image, display
    display(Image(figure))
except ImportError:
    pass

## 7. Circle of Willis labels
Development cases only: which labels exist and where the occlusion masks sit relative to them. This is the evidence on M1 versus M2.

In [ ]:
results['cow_lvo'] = survey.cow_lvo_survey(train_dir, development)
print(json.dumps(results['cow_lvo'], indent=1))

## 8. Values and candidate grids
A sample of development cases. Checks that NCCT is in HU and what units the maps use, then measures volume error and size for each spacing. This takes the longest of sections 4–8: allow 10–30 minutes.

In [ ]:
grid_sample = survey.sample_by_volume(volumes, development, GRID_CASES)
results['values'] = survey.value_survey(train_dir, grid_sample)
print(json.dumps(results['values'], indent=1), flush=True)
results['grid'] = survey.grid_candidates(train_dir, grid_sample, GRID_SPACINGS, n_all_cases=len(cases))
print(json.dumps(results['grid'], indent=1))

## 9. Save and check
Saves the results to `artifacts/isles24_survey/`, pushes them to Drive, reads them back, and confirms the archive is unchanged.

In [ ]:
heldout_set = set(folds['heldout'])
report_text = json.dumps(results)
leaked = sum(case in report_text for case in heldout_set - {r['case'] for r in results['extra_rows']} - {STRAY_CASE})
if leaked:
    raise ValueError(f'{leaked} held-out IDs reached the report. Nothing was saved. Stop and check.')

saved = os.path.join(OUT_DIR, 'isles24_survey.json')
with open(saved, 'w') as f:
    json.dump(results, f, indent=1)
shutil.copyfile(figure, os.path.join(OUT_DIR, 'spot_check.png'))
shutil.copyfile(os.path.join(LOCAL_DIR, 'infarct_volumes.json'), os.path.join(OUT_DIR, 'infarct_volumes.json'))
expected = json.load(open(saved))

drive.flush_and_unmount()
drive.mount('/content/drive', force_remount=True)

if json.load(open(saved)) != expected or not os.path.getsize(os.path.join(OUT_DIR, 'spot_check.png')):
    raise ValueError('What was read back from Drive differs from what was saved. Run this section again.')
after = os.stat(ARCHIVE)
if (after.st_size, after.st_mtime) != (before.st_size, before.st_mtime):
    raise ValueError('The archive size or modified time changed. Stop and check raw/.')
drive.flush_and_unmount()
print(f'OK: results saved to Drive and read back; archive unchanged; split sha256 {results["folds_sha256"][:12]}; development cases {len(development)}.')